In [ ]:
# CPU allocation controls. Edit these before running the notebook.
import os

CPU_START = int(os.environ.get("DWCONV_CPU_START", "0"))
CPU_COUNT = int(os.environ.get("DWCONV_CPU_COUNT", "20"))
N_JOBS = int(os.environ.get("DWCONV_N_JOBS", str(CPU_COUNT)))

os.environ["DWCONV_CPU_START"] = str(CPU_START)
os.environ["DWCONV_CPU_COUNT"] = str(CPU_COUNT)
os.environ["DWCONV_N_JOBS"] = str(N_JOBS)
os.environ["MY_CPU_COUNT"] = str(CPU_COUNT)
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["NUMEXPR_MAX_THREADS"] = "1"

requested = list(range(CPU_START, CPU_START + CPU_COUNT))
try:
    os.sched_setaffinity(0, set(requested))
except Exception as exc:
    print(f"CPU affinity not set: {exc}")
try:
    print("CPU affinity:", sorted(os.sched_getaffinity(0)))
except Exception:
    pass


# N20x20 DW-Truncation Convergence CPU Experiment

This notebook accompanies `run_n20x20_dwtrunc_convergence_cpu.py`. It can run the full CPU experiment or load the generated CSV outputs and regenerate the figures.

In [ ]:
from pathlib import Path
import sys

NOTEBOOK_DIR = Path.cwd()
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

import run_n20x20_dwtrunc_convergence_cpu as exp

OUTPUT_DIR = Path(exp.OUTPUT_DIR)
FIG_DIR = OUTPUT_DIR / "figures"
OUTPUT_DIR, FIG_DIR


In [ ]:
# Set True to run the full experiment from this notebook. The shell/tmux runner is preferred for long runs.
RUN_EXPERIMENT = False

if RUN_EXPERIMENT:
    exp.main()
else:
    print("Skipping run. Loading existing outputs from", OUTPUT_DIR)


In [ ]:
import json
import pandas as pd

ensemble_path = OUTPUT_DIR / "ensemble_cycle_curves.csv"
frob_path = OUTPUT_DIR / "frob_successive_deltas.csv"
metrics_path = OUTPUT_DIR / "per_sample_cycle_metrics.csv"
summary_path = OUTPUT_DIR / "comparison_summary.csv"
manifest_path = OUTPUT_DIR / "analysis_manifest.json"

missing = [p for p in [ensemble_path, frob_path, metrics_path, summary_path, manifest_path] if not p.exists()]
if missing:
    raise FileNotFoundError("Missing outputs. Run the script first: " + ", ".join(str(p) for p in missing))

ensemble = pd.read_csv(ensemble_path)
frob = pd.read_csv(frob_path)
metrics = pd.read_csv(metrics_path)
summary = pd.read_csv(summary_path)
manifest = json.loads(manifest_path.read_text())

summary


In [ ]:
import matplotlib.pyplot as plt

plt.rcParams.update({
    "font.family": exp.preferred_font_family(),
    "font.size": 8,
    "axes.titlesize": 8,
    "axes.labelsize": 8,
    "legend.fontsize": 7,
    "xtick.labelsize": 7,
    "ytick.labelsize": 7,
})
colors = {"dwtrunc0": "#4C72B0", "dwtrunc1": "#DD8452"}

fig, ax = plt.subplots(figsize=(3.375, 2.45), dpi=300)
for label, sub in ensemble.groupby("config_id"):
    sub = sub.sort_values("cycle")
    sub = sub[sub["cycle"] >= 1]
    ax.plot(sub["cycle"], sub["frob_successive_delta_mean"], marker="o", ms=2.2, lw=1.0, color=colors.get(label), label=label)
    ax.fill_between(sub["cycle"], sub["frob_successive_delta_mean"] - sub["frob_successive_delta_sem"], sub["frob_successive_delta_mean"] + sub["frob_successive_delta_sem"], color=colors.get(label), alpha=0.18)
ax.set_xlabel("cycle")
ax.set_ylabel(r"$\|G_c-G_{c-1}\|_F$")
ax.set_title("Successive covariance differences")
ax.grid(alpha=0.25, linewidth=0.5)
ax.legend(frameon=False)
fig.tight_layout()
fig.savefig(FIG_DIR / "frob_successive_delta_vs_cycle.png", dpi=300)
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(3.375, 2.45), dpi=300)
for label, sub in ensemble.groupby("config_id"):
    sub = sub.sort_values("cycle")
    ax.plot(sub["cycle"], sub["global_entropy_bits_mean"], marker="o", ms=2.2, lw=1.0, color=colors.get(label), label=label)
    ax.fill_between(sub["cycle"], sub["global_entropy_bits_mean"] - sub["global_entropy_bits_sem"], sub["global_entropy_bits_mean"] + sub["global_entropy_bits_sem"], color=colors.get(label), alpha=0.18)
ax.set_xlabel("cycle")
ax.set_ylabel(r"$S / \log 2$")
ax.set_title("Global entropy")
ax.grid(alpha=0.25, linewidth=0.5)
ax.legend(frameon=False)
fig.tight_layout()
fig.savefig(FIG_DIR / "global_entropy_vs_cycle.png", dpi=300)
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(6.75, 2.45), dpi=300)
for label, sub in ensemble.groupby("config_id"):
    sub = sub.sort_values("cycle")
    frob_sub = sub[sub["cycle"] >= 1]
    axes[0].plot(frob_sub["cycle"], frob_sub["frob_successive_delta_mean"], marker="o", ms=2.0, lw=1.0, color=colors.get(label), label=label)
    axes[0].fill_between(frob_sub["cycle"], frob_sub["frob_successive_delta_mean"] - frob_sub["frob_successive_delta_sem"], frob_sub["frob_successive_delta_mean"] + frob_sub["frob_successive_delta_sem"], color=colors.get(label), alpha=0.18)
    axes[1].plot(sub["cycle"], sub["global_entropy_bits_mean"], marker="o", ms=2.0, lw=1.0, color=colors.get(label), label=label)
    axes[1].fill_between(sub["cycle"], sub["global_entropy_bits_mean"] - sub["global_entropy_bits_sem"], sub["global_entropy_bits_mean"] + sub["global_entropy_bits_sem"], color=colors.get(label), alpha=0.18)
axes[0].set_xlabel("cycle")
axes[0].set_ylabel(r"$\|G_c-G_{c-1}\|_F$")
axes[0].set_title("Frobenius convergence")
axes[1].set_xlabel("cycle")
axes[1].set_ylabel(r"$S / \log 2$")
axes[1].set_title("Entropy convergence")
for ax in axes:
    ax.grid(alpha=0.25, linewidth=0.5)
    ax.legend(frameon=False)
fig.tight_layout()
fig.savefig(FIG_DIR / "convergence_comparison_panel.png", dpi=300)
plt.show()


In [ ]:
manifest.get("verdict", {})
